# Install Dependecies

In [1]:
%%capture
%pip install -q "transformers>=5,<6"
%pip install matplotlib
%pip install ipynbname
%pip install datasets
%pip install ipywidgets

In [2]:
%%capture
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install ipykernel

# Imports

In [46]:
import numpy as np
import pandas as pd
import random
import torch
import time
import math
import platform
import datasets
import transformers
from collections import Counter
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm.std import tqdm
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import SGDClassifier

# Import datasets

In [12]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

# Preprocessing
Each group member runs the experiment on one language.

In [13]:
LANGUAGE = "ko"

### 1. Questions
Only the questions are used.

In [14]:
# train
train_questions = df_train.loc[df_train["lang"] == LANGUAGE, "question"].drop_duplicates().tolist()
train_question_set = set(train_questions)

In [ ]:
# val
val_questions = df_val.loc[df_val["lang"] == LANGUAGE, "question"].drop_duplicates().tolist()

### 2. Tokenisation

In [9]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

In [10]:
BOS = "<s>"
EOS = "</s>"
UNK = "<UNK>"

In [11]:
# train
train_tokens = [tokenizer.tokenize(question) for question in train_questions]
# val
val_tokens = [tokenizer.tokenize(question) for question in val_questions]

print("train pieces:", sum(len(tokens) for tokens in train_tokens))
print("val pieces:", sum(len(tokens) for tokens in val_tokens))
print("XLM-R <unk> pieces in val:", sum(tokens.count(tokenizer.unk_token) for tokens in val_tokens))

train pieces: 29090
val pieces: 3520
XLM-R <unk> pieces in val: 0


### 3. Unknown-token strategy
The language model only knows the pieces it saw in the training questions. Pieces that occur fewer than `MIN_COUNT` times in training are replaced by `<UNK>`, so the model learns a probability for rare pieces. Any validation piece that is not in the vocabulary is also mapped to `<UNK>`.

We report the `<UNK>` rate, because a high rate makes perplexity look better than it is: all rare pieces share one frequent `<UNK>` token.

In [16]:
# claude
MIN_COUNT = 2

train_piece_counts = Counter(piece for tokens in train_tokens for piece in tokens)
vocab = {piece for piece, count in train_piece_counts.items() if count >= MIN_COUNT} | {UNK, EOS}
print("distinct train pieces:", len(train_piece_counts), "| vocab size (with <UNK> and </s>):", len(vocab))

distinct train pieces: 2223 | vocab size (with <UNK> and </s>): 1521


In [17]:
# claude
def replace_unknown(tokens):
    return [piece if piece in vocab else UNK for piece in tokens]

In [18]:
# claude
def unk_rate(token_lists):
    pieces = [piece for tokens in token_lists for piece in tokens]
    return pieces.count(UNK) / len(pieces)

In [19]:
# claude
# train
train_tokens_unk = [replace_unknown(tokens) for tokens in train_tokens]
# val
val_tokens_unk = [replace_unknown(tokens) for tokens in val_tokens]

print("train <UNK> rate:", round(unk_rate(train_tokens_unk), 3))
print("val <UNK> rate:", round(unk_rate(val_tokens_unk), 3))

train <UNK> rate: 0.024
val <UNK> rate: 0.05


### 4. N-gram language models
We compare three models that use the same pieces and the same vocabulary:
* unigram (baseline)
* bigram
* trigram

Probabilities are relative counts from the training questions (maximum likelihood). The bigram and trigram models use linear interpolation with the lower orders, so an n-gram that was never seen in training still gets a probability:

$$P(w \mid u, v) = \lambda_1 P(w) + \lambda_2 P(w \mid v) + \lambda_3 P(w \mid u, v)$$

If the history $(u, v)$ was never seen in training, the trigram term uses the bigram estimate instead (and the same for an unseen $v$), so the probabilities always sum to 1. The unigram model is $\lambda = (1, 0, 0)$ and a bigram model has $\lambda_3 = 0$.

In [20]:
# claude
# counts every unigram, bigram and trigram that ends in a predicted piece
def count_ngrams(token_lists):
    counts = Counter()
    for tokens in token_lists:
        padded = [BOS, BOS] + tokens + [EOS]
        for i in range(2, len(padded)):
            counts[(padded[i],)] += 1
            counts[(padded[i - 1], padded[i])] += 1
            counts[(padded[i - 2], padded[i - 1], padded[i])] += 1
    return counts

In [21]:
# claude
# how often each history is followed by a piece; the unigram history is ()
def count_contexts(ngram_counts):
    contexts = Counter()
    for ngram, count in ngram_counts.items():
        contexts[ngram[:-1]] += count
    return contexts

In [24]:
# claude
# train
start = time.perf_counter()
ngram_counts = count_ngrams(train_tokens_unk)
context_counts = count_contexts(ngram_counts)
train_time = time.perf_counter() - start

print("training time (s):", round(train_time, 3))
print("distinct n-grams by order:", sorted(Counter(len(ngram) for ngram in ngram_counts).items()))

training time (s): 0.03
distinct n-grams by order: [(1, 1521), (2, 9958), (3, 15320)]


In [25]:
# claude
def mle(ngram):
    return ngram_counts[ngram] / context_counts[ngram[:-1]]

In [26]:
# claude
# probability of piece w after the history (u, v)
def interpolated_prob(u, v, w, lambdas):
    l1, l2, l3 = lambdas
    p1 = mle((w,))
    p2 = mle((v, w)) if context_counts[(v,)] > 0 else p1
    p3 = mle((u, v, w)) if context_counts[(u, v)] > 0 else p2
    return l1 * p1 + l2 * p2 + l3 * p3

In [29]:
# claude
# log probability of every predicted piece in one question (including </s>)
def question_log_probs(tokens, lambdas):
    padded = [BOS, BOS] + tokens + [EOS]
    return [math.log(interpolated_prob(padded[i - 2], padded[i - 1], padded[i], lambdas)) for i in range(2, len(padded))]

In [30]:
# claude
# per-token perplexity over all questions: exp of the mean negative log probability
def perplexity(token_lists, lambdas):
    log_probs = [log_prob for tokens in token_lists for log_prob in question_log_probs(tokens, lambdas)]
    return math.exp(-sum(log_probs) / len(log_probs))

### 5. Automatic checks
1. For every history, the probabilities over the vocabulary sum to 1, also when the history was never seen in training.
2. A piece that is not in the vocabulary is mapped to `<UNK>`.
3. `<s>` is never predicted.

In [31]:
# claude
# check 1: probabilities sum to 1
assert {ngram[0] for ngram in ngram_counts if len(ngram) == 1} == vocab
histories = [
    (BOS, BOS),                           # start of a question
    (BOS, val_tokens_unk[0][0]),          # after the first piece of a val question
    (UNK, UNK),                           # two rare pieces
    (BOS, EOS),                           # never seen in training
]
for u, v in histories:
    for lambdas in [(1.0, 0.0, 0.0), (0.2, 0.8, 0.0), (0.1, 0.3, 0.6)]:
        total = sum(interpolated_prob(u, v, w, lambdas) for w in vocab)
        assert abs(total - 1) < 1e-9, (u, v, lambdas, total)
print("check 1: probabilities sum to 1")

# check 2: unseen piece
assert replace_unknown(["▁piece_not_in_training"]) == [UNK]
print("check 2: unseen piece is mapped to <UNK>")

# check 3: <s> is only used as history
assert (BOS,) not in ngram_counts
print("check 3: <s> is never predicted")

check 1: probabilities sum to 1
check 2: unseen piece is mapped to <UNK>
check 3: <s> is never predicted


### 6. Choose λ on validation
Following Section 1, the interpolation weights are selected on the validation split. We try a small grid; rows with λ3 = 0 are bigram models. Because the same split is used for selection and reporting, the reported validation perplexity is slightly optimistic.

In [32]:
# claude
rows = []
for l1 in [0.1, 0.2, 0.3, 0.4, 0.5]:
    for l3 in [0.0, 0.2, 0.4, 0.6, 0.8]:
        l2 = round(1 - l1 - l3, 1)
        if l2 < 0:
            continue
        rows.append({"l1": l1, "l2": l2, "l3": l3, "val_ppl": perplexity(val_tokens_unk, (l1, l2, l3))})

grid = pd.DataFrame(rows).sort_values("val_ppl")
display(grid.head(10))

,l1,l2,l3,val_ppl
8,0.2,0.2,0.6,23.675046
7,0.2,0.4,0.4,24.005970
12,0.3,0.3,0.4,24.137226
13,0.3,0.1,0.6,24.192274
3,0.1,0.3,0.6,24.948770
16,0.4,0.2,0.4,25.169001
6,0.2,0.6,0.2,25.408818
2,0.1,0.5,0.4,25.484719
11,0.3,0.5,0.2,25.510748
4,0.1,0.1,0.8,25.686581


In [33]:
# claude
best_bigram = grid[grid["l3"] == 0].iloc[0]
best_trigram = grid[grid["l3"] > 0].iloc[0]

MODELS = {
    "unigram": (1.0, 0.0, 0.0),
    "bigram": (float(best_bigram["l1"]), float(best_bigram["l2"]), 0.0),
    "trigram": (float(best_trigram["l1"]), float(best_trigram["l2"]), float(best_trigram["l3"])),
}
print(MODELS)

{'unigram': (1.0, 0.0, 0.0), 'bigram': (0.2, 0.8, 0.0), 'trigram': (0.2, 0.2, 0.6)}


### 7. Results
Per-token perplexity (lower is better), with the number of evaluated questions and tokens. Tokens are XLM-R pieces plus one `</s>` per question. `val_ppl_not_in_train` leaves out the validation questions that also occur in the training questions.

In [34]:
# claude
# number of predicted tokens: every piece plus one </s> per question
def n_predicted(token_lists):
    return sum(len(tokens) + 1 for tokens in token_lists)

In [35]:
# claude
val_tokens_not_in_train = [tokens for question, tokens in zip(val_questions, val_tokens_unk) if question not in train_question_set]

rows = []
for name, lambdas in MODELS.items():
    rows.append({
        "model": name,
        "lambdas": lambdas,
        "train_ppl": round(perplexity(train_tokens_unk, lambdas), 2),
        "val_ppl": round(perplexity(val_tokens_unk, lambdas), 2),
        "val_ppl_not_in_train": round(perplexity(val_tokens_not_in_train, lambdas), 2),
    })
display(pd.DataFrame(rows))

print("train:", len(train_tokens_unk), "questions,", n_predicted(train_tokens_unk), "tokens")
print("val:", len(val_tokens_unk), "questions,", n_predicted(val_tokens_unk), "tokens")
print("val not in train:", len(val_tokens_not_in_train), "questions,", n_predicted(val_tokens_not_in_train), "tokens")

,model,lambdas,train_ppl,val_ppl,val_ppl_not_in_train
0,unigram,"(1.0, 0.0, 0.0)",245.58,231.89,232.34
1,bigram,"(0.2, 0.8, 0.0)",9.67,30.07,31.42
2,trigram,"(0.2, 0.2, 0.6)",3.70,23.68,25.15


train: 2397 questions, 31487 tokens
val: 292 questions, 3812 tokens
val not in train: 282 questions, 3686 tokens


### 8. High and low perplexity questions
Per-question perplexity with the trigram model.

In [36]:
# claude
def question_perplexity(tokens, lambdas):
    log_probs = question_log_probs(tokens, lambdas)
    return math.exp(-sum(log_probs) / len(log_probs))

In [37]:
# claude
df_ppl = pd.DataFrame({
    "question": val_questions,
    "n_pieces": [len(tokens) for tokens in val_tokens_unk],
    "n_unk": [tokens.count(UNK) for tokens in val_tokens_unk],
    "in_train": [question in train_question_set for question in val_questions],
    "ppl": [question_perplexity(tokens, MODELS["trigram"]) for tokens in val_tokens_unk],
}).sort_values("ppl")

In [38]:
# claude
# lowest perplexity
display(df_ppl.head(10))

,question,n_pieces,n_unk,in_train,ppl
50,개혁침례교는 신교와 상관관계가있는가?,16,0,True,2.824444
263,중국에서 가장 인구밀도가 높은 도시는 어디인가?,12,0,False,2.885226
173,스웨덴에서 가장 인구밀도가 높은 도시는 어디인가?,14,0,False,3.116748
85,일본에서 가장 많이 판매된 만화책은 무엇인가?,11,0,False,3.572625
187,에피쿠로스는 언제 태어났나요?,11,0,True,3.582918
252,1975년 미국의 지도자는 누구인가?,8,1,False,3.616090
123,독일의 연방 수상은 누구인가요?,10,0,True,3.692741
198,중국에서 가장 작은 도시는 어디인가?,9,0,False,3.716481
237,노르망디주에는 몇 개의 초등학교가 있나요?,14,0,False,3.819459
87,인간의 몸에서 가장 큰 뼈는 무엇인가?,11,0,True,3.874927


In [39]:
# claude
# highest perplexity
display(df_ppl.tail(10))

,question,n_pieces,n_unk,in_train,ppl
178,클라미디아 발병원인은 무엇인가?,9,1,False,235.030730
248,인체가 자외선에 많이 노출된다면 어떤 변화가 일어나는가?,18,3,False,265.188830
121,근대의 식민지는 어느 시대에 본격화 되었나요?,13,1,False,278.267461
195,코탕탱반도는 현재의 어디인가?,10,0,False,280.093999
251,"임시정부는 종전의 정부가 무너진 후, 무정부 상태를 해소하기 위해 임시로 구성된 정...",32,4,False,293.467119
120,배관공이 하는 일은 무엇인가?,9,0,False,337.268724
104,제1대 휘트워스 준남작 조지프 휘트워스 경의 소속은 어디인가요?,27,2,False,350.524146
24,육파철학의 이름들은 무엇인가?,9,0,False,443.722721
181,불고기는 소의 어느부위를 사용하는가?,11,0,False,456.403166
52,더글러스 맥아더 배우자는 누군가요?,12,0,False,488.202585


In [40]:
# claude
def piece_probs(question, lambdas):
    tokens = replace_unknown(tokenizer.tokenize(question))
    padded = [BOS, BOS] + tokens + [EOS]
    return pd.DataFrame({
        "piece": padded[2:],
        "prob": [interpolated_prob(padded[i - 2], padded[i - 1], padded[i], lambdas) for i in range(2, len(padded))],
    })

In [41]:
# claude
# lowest perplexity
question = df_ppl.iloc[0]["question"]
print(question)
display(piece_probs(question, MODELS["trigram"]))

개혁침례교는 신교와 상관관계가있는가?


,piece,prob
0,▁,0.117999
1,개혁,0.004223
2,침,0.500013
3,례,0.800019
4,교,0.733594
5,는,0.375559
6,▁신,0.043184
7,교,0.608594
8,와,0.605024
9,▁상,0.608772


In [42]:
# claude
# highest perplexity
question = df_ppl.iloc[-1]["question"]
print(question)
display(piece_probs(question, MODELS["trigram"]))

더글러스 맥아더 배우자는 누군가요?


,piece,prob
0,▁더,0.000025
1,글,0.000032
2,러,0.000070
3,스,0.000597
4,▁맥,0.000025
5,아,0.000254
6,더,0.000032
7,▁배우,0.000044
8,자는,0.228991
9,▁누군가,0.303113


### 9. Hypothesis check
Numbers to test a hypothesis about what the model captures: does per-question perplexity go with question length, with the share of `<UNK>` pieces, or with having seen the question in training?

In [43]:
# claude
df_ppl["unk_share"] = df_ppl["n_unk"] / df_ppl["n_pieces"]
print(df_ppl[["ppl", "n_pieces", "unk_share"]].corr(method="spearman")["ppl"])
display(df_ppl.groupby("in_train")["ppl"].agg(["count", "median"]))

ppl          1.000000
n_pieces     0.180324
unk_share    0.278367
Name: ppl, dtype: float64


,count,median
in_train,,
False,282,21.359005
True,10,4.283581


### 10. Reproducibility
Hardware, software versions, training time and number of training examples. The n-gram counts are deterministic, so no random seed is involved.

In [47]:
# claude
print("python:", platform.python_version())
print("platform:", platform.platform())
print("processor:", platform.processor())
print("pandas:", pd.__version__)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("training time (s):", round(train_time, 3))
print("training questions:", len(train_questions), "| training tokens:", n_predicted(train_tokens_unk))

python: 3.14.6
platform: Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.43
processor: 
pandas: 2.3.3
datasets: 4.8.5
transformers: 5.16.1
training time (s): 0.03
training questions: 2397 | training tokens: 31487
